In [ ]:
## TOAST EXAMPLE NOTEBOOK ##

In [1]:
import matplotlib
import astropy.units as u
from astropy.coordinates import SkyCoord
from toast import QueryWidget # the on-sky area stacking tool
import numpy as np
import matplotlib.pyplot as plt

In [2]:
%matplotlib widget

In [12]:
# need if querying by Detectid, be mindful of memory usage. If possible, I recommend querying by coordinate
# note this specific one is the line detections catalog, so will only search those ones. 
from hetdex_api.detections import Detections
detects = Detections(curated_version='5.0.1')

In [15]:
### Main Widget Usage ###

# Usage order: Lasso Select --> Confirm Selection --> Extract Spectra (--> Reset)
try:
    qw.close()
except:
    pass

qw = QueryWidget(#coords = SkyCoord(236.8223419*u.deg, 49.0525512* u.deg, frame="icrs"), 
                 detectid = 5003879008, 
                 detections = detects, # this must also be on if querying by detectid
                 cutout_size=15*u.arcsec, 
                 spec_mode='fibers', # 'fibers', 'psf'
                # wave_range = (4060, 4340),
                # flux_range = (-0.3, 0.4),
                 stat='median', # 'mean', 'median', 'biweight', 'weighted_biweight'
                 )
# Image loading is not always immediate. matplotlib has to redraw the figure for every reset/pan/lasso draw. please be patient. 
# To select different fibers, simply redraw the lasso or hit reset. However, the reset button will pan back to original coords. 

# NOTE: if you use the up/down arrows on RA/Dec, it will truncate to the set step (i.e. be less exact). 
# The exact selected fiber coordinates are saved in the output table.

# if the loaded image says "SDSS query", we do not have imaging and the code attempts an SDSS search. TODO: this comes out wonky. fix. 
# if it does not find by detectid, search by coordinate. that detectid may not be in the version specified to Detections class. 

In [16]:
from hetdex_api.elixer_widget_cls import ElixerWidget
EW = ElixerWidget(detectlist=[5003879008])

interactive(children=(Text(value='5003879008', description='DetectID', placeholder='5003879008'), Output()), _…

In [ ]:
fiber_stacked_spec = qw.stacked_spec

In [ ]:
CALFIB_WAVEGRID = np.linspace(3470,5540,1036)

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5.5, 4))
ax.plot(CALFIB_WAVEGRID, fiber_stacked_spec, label='fiber staked spectrum')
ax.set_xlabel(r'observed wavelength ($\AA$)')
ax.set_ylabel(r'$\mathrm{f_{\lambda}~(10^{-17}~ergs/s/cm^2/\AA)}$')
plt.legend()
plt.show()

In [ ]:
"""
Ideas for additions:
- option to set zscale manually, best for nearby extended sources, not super useful to typical HETDEX LAEs.
- add color plotted to output table (must select less than 10 fibers)
"""